In [1]:
# 01_data_cleaning.ipynb — Cell 1: Load & Inspect
import pandas as pd
import numpy as np
# Run once — sets working directory to project root regardless of where notebook is
import os
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir('..')
print('Working directory:', os.getcwd())

df = pd.read_csv('data/raw/cms_raw.csv', dtype=str, encoding='latin-1')  # load ALL as strings first

# ── Five essential first looks ──────────────────────────────
print('Shape:', df.shape)
print('\nColumn names:')
print(df.columns.tolist())
print('\nNull counts:')
print(df.isnull().sum())
print('\nSample rows:')
print(df.head(3).to_string())
print('\nUnique providers:')
print(df['Rndrng_Prvdr_CCN'].nunique())


Working directory: c:\Users\gamer\Documents\projects\cms-medicare-audit-analysis
Shape: (146427, 15)

Column names:
['Rndrng_Prvdr_CCN', 'Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City', 'Rndrng_Prvdr_St', 'Rndrng_Prvdr_State_FIPS', 'Rndrng_Prvdr_Zip5', 'Rndrng_Prvdr_State_Abrvtn', 'Rndrng_Prvdr_RUCA', 'Rndrng_Prvdr_RUCA_Desc', 'DRG_Cd', 'DRG_Desc', 'Tot_Dschrgs', 'Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']

Null counts:
Rndrng_Prvdr_CCN             0
Rndrng_Prvdr_Org_Name        0
Rndrng_Prvdr_City            0
Rndrng_Prvdr_St              0
Rndrng_Prvdr_State_FIPS      0
Rndrng_Prvdr_Zip5            0
Rndrng_Prvdr_State_Abrvtn    0
Rndrng_Prvdr_RUCA            4
Rndrng_Prvdr_RUCA_Desc       4
DRG_Cd                       0
DRG_Desc                     0
Tot_Dschrgs                  0
Avg_Submtd_Cvrd_Chrg         0
Avg_Tot_Pymt_Amt             0
Avg_Mdcr_Pymt_Amt            0
dtype: int64

Sample rows:
  Rndrng_Prvdr_CCN            Rndrng_Prvdr_Org_Name Rndrng_Prvdr_C

In [2]:
# Cell 2: Fix currency columns
# ── Identify all currency columns ───────────────────────────
currency_cols = [
    'Avg_Submtd_Cvrd_Chrg',   # what the hospital charged
    'Avg_Tot_Pymt_Amt',        # total payment (Medicare + other payers)
    'Avg_Mdcr_Pymt_Amt',       # Medicare's portion of the payment
]
# ── Strip and coerce ─────────────────────────────────────────
for col in currency_cols:
    # Remove dollar sign and commas, then convert
    df[col] = (df[col]
               .str.replace('$', '', regex=False)
               .str.replace(',', '', regex=False)
               .str.strip())
    df[col] = pd.to_numeric(df[col], errors='coerce')

# ── Validate: how many nulls did coercion create? ────────────
print('Nulls after coercion:')
print(df[currency_cols].isnull().sum())
# ── Document any unexpected nulls ───────────────────────────
# If a column has >0 new nulls, investigate before dropping
unexpected = df[df['Avg_Submtd_Cvrd_Chrg'].isnull()]
print(f'Rows with unparseable charges: {len(unexpected)}')
print(unexpected[['Rndrng_Prvdr_Org_Name','Avg_Submtd_Cvrd_Chrg']].head())


Nulls after coercion:
Avg_Submtd_Cvrd_Chrg    0
Avg_Tot_Pymt_Amt        0
Avg_Mdcr_Pymt_Amt       0
dtype: int64
Rows with unparseable charges: 0
Empty DataFrame
Columns: [Rndrng_Prvdr_Org_Name, Avg_Submtd_Cvrd_Chrg]
Index: []


In [3]:
# Cell 3: Fix identifier types
# ── Provider CCN: strip .0 and zero-pad to 6 digits ─────────
# CMS Certification Numbers are always 6 digits with leading zeros
df['Rndrng_Prvdr_CCN'] = (
    df['Rndrng_Prvdr_CCN']
    .str.split('.').str[0]          # remove .0 suffix
    .str.strip()
    .str.zfill(6)                   # pad to 6 digits e.g. '12345' → '012345'
)

# ── Validate: check a sample for correct format ─────────────
print('CCN sample:', df['Rndrng_Prvdr_CCN'].head(10).tolist())
print('CCN length check:', df['Rndrng_Prvdr_CCN'].str.len().value_counts())

# ── ZIP code: same approach, pad to 5 digits ─────────────────
df['Rndrng_Prvdr_Zip5'] = (
    df['Rndrng_Prvdr_Zip5']
    .str.split('.').str[0]
    .str.strip()
    .str.zfill(5)
)

# Flag any ZIP that isn't exactly 5 digits after cleaning
bad_zips = df[df['Rndrng_Prvdr_Zip5'].str.len() != 5]
print(f'Non-standard ZIPs remaining: {len(bad_zips)}')

CCN sample: ['010001', '010001', '010001', '010001', '010001', '010001', '010001', '010001', '010001', '010001']
CCN length check: Rndrng_Prvdr_CCN
6    146427
Name: count, dtype: int64
Non-standard ZIPs remaining: 0


In [4]:
# Cell 4: Standardise text columns
# ── Step 1: strip whitespace and normalise case ─────────────
text_cols = ['Rndrng_Prvdr_Org_Name', 'Rndrng_Prvdr_City',
             'Rndrng_Prvdr_State_Abrvtn', 'DRG_Desc']

for col in text_cols:
    df[col] = df[col].str.strip().str.upper()

# ── Step 2: standardise common abbreviations in names ────────
name_replacements = {
    r'\bST\.?\b':       'SAINT',
    r'\bMED\.?\b':      'MEDICAL',
    r'\bHOSP\.?\b':     'HOSPITAL',
    r'\bMEML?\.?\b':    'MEMORIAL',
    r'\bREGL?\.?\b':    'REGIONAL',
    r'\bCTR\.?\b':      'CENTER',
    r'\bUNIV\.?\b':     'UNIVERSITY',
}
for pattern, replacement in name_replacements.items():
    df['Rndrng_Prvdr_Org_Name'] = (
        df['Rndrng_Prvdr_Org_Name']
        .str.replace(pattern, replacement, regex=True)
    )

# ── Step 3: fix DRG_Desc trailing spaces (source of false dupes)
df['DRG_Desc'] = df['DRG_Desc'].str.strip()

# ── Step 4: check state column for non-standard values ───────
valid_states = ['AL','AK','AZ','AR','CA','CO','CT','DE','FL','GA',
                'HI','ID','IL','IN','IA','KS','KY','LA','ME','MD',
                'MA','MI','MN','MS','MO','MT','NE','NV','NH','NJ',
                'NM','NY','NC','ND','OH','OK','OR','PA','RI','SC',
                'SD','TN','TX','UT','VT','VA','WA','WV','WI','WY',
                'DC','GU','PR','VI']
bad_states = df[~df['Rndrng_Prvdr_State_Abrvtn'].isin(valid_states)]
print(f'Non-standard state values: {len(bad_states)}')
print(bad_states['Rndrng_Prvdr_State_Abrvtn'].value_counts().head(10))


Non-standard state values: 0
Series([], Name: count, dtype: int64)


In [5]:
# Cell 5: Duplicate detection
# ── Define the natural key for this dataset ─────────────────
# One row should = one provider × one DRG in one year
key_cols = ['Rndrng_Prvdr_CCN', 'DRG_Cd']

# ── Check for true duplicates ────────────────────────────────
n_dupes = df.duplicated(subset=key_cols, keep=False).sum()
print(f'Rows involved in duplicate key pairs: {n_dupes}')

# ── Inspect them before deciding what to do ──────────────────
dupes = df[df.duplicated(subset=key_cols, keep=False)].sort_values(key_cols)
print(dupes[key_cols + ['Tot_Dschrgs','Avg_Submtd_Cvrd_Chrg']].head(20))

# ── Decision logic ───────────────────────────────────────────
# If charges are IDENTICAL → pure duplicate, keep first occurrence
# If charges DIFFER → possible amended filing, flag and keep both with a note

charge_cols = ['Avg_Submtd_Cvrd_Chrg', 'Avg_Mdcr_Pymt_Amt']
identical_dupes = df.duplicated(subset=key_cols + charge_cols, keep=False)
df['is_duplicate'] = identical_dupes

print(f'Identical duplicate rows: {identical_dupes.sum()}')

# Remove identical duplicates, keep first
df_clean = df[~df.duplicated(subset=key_cols + charge_cols, keep='first')].copy()
print(f'Rows after dedup: {len(df_clean)} (was {len(df)})')


Rows involved in duplicate key pairs: 0
Empty DataFrame
Columns: [Rndrng_Prvdr_CCN, DRG_Cd, Tot_Dschrgs, Avg_Submtd_Cvrd_Chrg]
Index: []
Identical duplicate rows: 0
Rows after dedup: 146427 (was 146427)


In [6]:
# Cell 6A: Logical outlier detection
# Under Medicare IPPS, what a hospital is paid cannot exceed what it charged.
# Rows where payment > charge are logically invalid.

df_clean['charge_to_payment_ratio'] = (
    df_clean['Avg_Submtd_Cvrd_Chrg'] / df_clean['Avg_Mdcr_Pymt_Amt']
)

# Flag rows where ratio < 1.0 (paid more than charged — impossible)
df_clean['logical_error_flag'] = df_clean['charge_to_payment_ratio'] < 1.0
logical_errors = df_clean['logical_error_flag'].sum()
print(f'Logically invalid rows (payment > charge): {logical_errors}')

# Separate them for investigation — do NOT silently drop
df_errors   = df_clean[df_clean['logical_error_flag']].copy()
df_analysis = df_clean[~df_clean['logical_error_flag']].copy()

print('Logical errors by state:')
print(df_errors['Rndrng_Prvdr_State_Abrvtn'].value_counts().head(10))
df_errors.to_csv('outputs/logical_error_rows.csv', index=False)

Logically invalid rows (payment > charge): 176
Logical errors by state:
Rndrng_Prvdr_State_Abrvtn
MD    48
IL    25
NY    25
TX    23
MA    16
AZ     9
MO     6
AL     3
FL     3
MS     3
Name: count, dtype: int64


In [7]:
# Cell 6B: Statistical outlier detection
# Statistical outliers: hospitals with unusually high or low charge-to-payment ratios
# Use IQR method — robust to skewed distributions common in healthcare billing data

Q1 = df_analysis['charge_to_payment_ratio'].quantile(0.25)
Q3 = df_analysis['charge_to_payment_ratio'].quantile(0.75)
IQR = Q3 - Q1

lower_bound = max(Q1 - 1.5 * IQR, 1.0)  # floor at 1.0 — ratios below 1 are logical errors
upper_bound = Q3 + 1.5 * IQR

print(f'IQR bounds: [{lower_bound:.2f}, {upper_bound:.2f}]')
print(f'Median ratio: {df_analysis["charge_to_payment_ratio"].median():.2f}')

df_analysis['ratio_outlier_flag'] = (
    (df_analysis['charge_to_payment_ratio'] < lower_bound) |
    (df_analysis['charge_to_payment_ratio'] > upper_bound)
)

n_outliers = df_analysis['ratio_outlier_flag'].sum()
pct = n_outliers / len(df_analysis) * 100
print(f'Statistical outliers: {n_outliers} rows ({pct:.1f}%)')

IQR bounds: [1.00, 13.12]
Median ratio: 5.52
Statistical outliers: 7840 rows (5.4%)


Finding: No nulls in any payment or charge columns. CMS appears to have excluded low-volume DRG records (typically suppressed when fewer than 11 discharges) from this data release. No suppression flag required.

In [8]:
# Cell 7: Null strategy
# ── Implement the null strategy ──────────────────────────────

# No nulls found in any money columns after cleaning
# CMS appears to have excluded suppressed records (<11 patients) from this release
# is_suppressed flag not needed

# 1. Flag suppressed charge rows before any analysis
print('Money column nulls:')
print(df_analysis[['Avg_Submtd_Cvrd_Chrg', 'Avg_Tot_Pymt_Amt', 'Avg_Mdcr_Pymt_Amt']].isnull().sum())
#df_analysis['is_suppressed'] = df_analysis['Avg_Mdcr_Pymt_Amt'].isnull()

# 2. Fill missing ZIP with placeholder
df_analysis['Rndrng_Prvdr_Zip5'] = (
    df_analysis['Rndrng_Prvdr_Zip5'].fillna('UNKNOWN')
)

# 3. Drop rows where key metric (discharges) is null — unusable
before = len(df_analysis)
df_analysis = df_analysis.dropna(subset=['Tot_Dschrgs'])
print(f'Dropped {before - len(df_analysis)} rows with null discharges')

# 4. Drop rows where BOTH charge and payment are null
#both_null = df_analysis[['Avg_Submtd_Cvrd_Chrg', 'Avg_Mdcr_Pymt_Amt']].isnull().all(axis=1)
#df_analysis = df_analysis[~both_null]
#print(f'Dropped {both_null.sum()} rows with no charge or payment data')

# ── Final null report ────────────────────────────────────────
print('\nNull counts in clean dataset:')
null_counts = df_analysis.isnull().sum()
print(null_counts[null_counts > 0])

Money column nulls:
Avg_Submtd_Cvrd_Chrg    0
Avg_Tot_Pymt_Amt        0
Avg_Mdcr_Pymt_Amt       0
dtype: int64
Dropped 0 rows with null discharges

Null counts in clean dataset:
Rndrng_Prvdr_RUCA         4
Rndrng_Prvdr_RUCA_Desc    4
dtype: int64


In [9]:
# Cell 8: Validation and export
# ── Final shape and sanity checks ───────────────────────────
print('Final dataset shape:', df_analysis.shape)
print('Unique providers:', df_analysis['Rndrng_Prvdr_CCN'].nunique())
print('Unique DRGs:', df_analysis['DRG_Cd'].nunique())
print('States covered:', sorted(df_analysis['Rndrng_Prvdr_State_Abrvtn'].unique()))

# ── Ratio summary statistics ──────────────────────────────────
print('\nCharge-to-payment ratio stats:')
print(df_analysis['charge_to_payment_ratio'].describe().round(2))

# ── Check no logical errors remain ───────────────────────────
assert (df_analysis['charge_to_payment_ratio'] >= 1.0).all(), \
    'Logical error rows remain — check Step 6A'

# ── Export ───────────────────────────────────────────────────
df_analysis.to_csv('data/processed/cms_clean.csv', index=False)
print('\n✅  Saved: data/processed/cms_clean.csv')

# ── Cleaning summary for README ──────────────────────────────
summary = f'''
Cleaning Summary
---------------------------------------
Original rows:     {len(df):,}
After dedup:       (see Step 5)
Logical errors:    {df_errors["logical_error_flag"].sum():,} rows separated
Final clean rows:  {len(df_analysis):,}
Statistical flags: {df_analysis["ratio_outlier_flag"].sum():,} outlier rows flagged
'''
print(summary)

Final dataset shape: (146251, 19)
Unique providers: 2937
Unique DRGs: 534
States covered: ['AK', 'AL', 'AR', 'AZ', 'CA', 'CO', 'CT', 'DC', 'DE', 'FL', 'GA', 'HI', 'IA', 'ID', 'IL', 'IN', 'KS', 'KY', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'MS', 'MT', 'NC', 'ND', 'NE', 'NH', 'NJ', 'NM', 'NV', 'NY', 'OH', 'OK', 'OR', 'PA', 'RI', 'SC', 'SD', 'TN', 'TX', 'UT', 'VA', 'VT', 'WA', 'WI', 'WV', 'WY']

Charge-to-payment ratio stats:
count    146251.00
mean          6.34
std           3.61
min           1.00
25%           4.05
50%           5.52
75%           7.68
max         141.52
Name: charge_to_payment_ratio, dtype: float64

✅  Saved: data/processed/cms_clean.csv

Cleaning Summary
---------------------------------------
Original rows:     146,427
After dedup:       (see Step 5)
Logical errors:    176 rows separated
Final clean rows:  146,251
Statistical flags: 7,840 outlier rows flagged

